# Run the PacBio HiFi 16S Nextflow pipeline

This notebook validates the project inputs, builds a reproducible command, and optionally runs the local `HiFi-16S-workflow`. Pipeline execution is disabled by default to prevent an accidental multi-hour rerun.

In [ ]:
from pathlib import Path
import csv
import shlex
import shutil
import subprocess

PROJECT_ROOT = Path('/home/rare/arlen/Pacbio_colombian_vaginal_microbiome')
PIPELINE_DIR = PROJECT_ROOT / 'HiFi-16S-workflow'
SAMPLES_TSV = PROJECT_ROOT / 'data/input_files/samples.tsv'
METADATA_TSV = PROJECT_ROOT / 'data/input_files/metadata.tsv'
OUTDIR = PROJECT_ROOT / 'results_HiFi-16S-workflow/01_nextflow_out'

PROFILE = 'conda'
DADA2_CPUS = 16
VSEARCH_CPUS = 16
RESUME = True
RUN_PIPELINE = False  # Set to True only when ready to execute.

## Validate software, workflow, samples, and metadata

In [ ]:
required = [PIPELINE_DIR / 'main.nf', PIPELINE_DIR / 'nextflow.config', SAMPLES_TSV, METADATA_TSV]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))
if shutil.which('nextflow') is None:
    raise RuntimeError('nextflow is not available on PATH')

def read_tsv(path):
    with path.open(newline='') as handle:
        return list(csv.DictReader(handle, delimiter='\t'))

samples = read_tsv(SAMPLES_TSV)
metadata = read_tsv(METADATA_TSV)
if not samples or not {'sample-id', 'absolute-filepath'} <= set(samples[0]):
    raise ValueError('samples.tsv must contain sample-id and absolute-filepath')
metadata_id_column = 'sample-id' if metadata and 'sample-id' in metadata[0] else 'sample_name'
if not metadata or metadata_id_column not in metadata[0] or 'condition' not in metadata[0]:
    raise ValueError('metadata.tsv must contain a sample ID column and condition')

sample_ids = [row['sample-id'] for row in samples]
metadata_ids = {row[metadata_id_column] for row in metadata}
missing_fastq = [row['absolute-filepath'] for row in samples if not Path(row['absolute-filepath']).is_file()]
missing_metadata = sorted(set(sample_ids) - metadata_ids)
print(f'Samples: {len(samples)}; metadata rows: {len(metadata)}')
print(f'FASTQ paths currently missing: {len(missing_fastq)}')
print(f'Samples missing metadata: {len(missing_metadata)}')
if missing_fastq:
    print('The manifest contains paths from another machine. Regenerate it with the next cell before running.')

## Repair the sample manifest when paths have moved

Run this cell if the validation reports missing FASTQs. It rewrites only the in-memory command input by creating `samples_local.tsv` beside this notebook; the original manifest is preserved.

In [ ]:
LOCAL_SAMPLES_TSV = Path.cwd() / 'samples_local.tsv'
FASTQ_DIR = PROJECT_ROOT / 'data/input_files'
local_rows = []
for sample_id in sample_ids:
    matches = sorted(FASTQ_DIR.glob(f'{sample_id}*.fastq.gz'))
    if len(matches) != 1:
        raise ValueError(f'Expected one FASTQ for {sample_id}, found {len(matches)}')
    local_rows.append({'sample-id': sample_id, 'absolute-filepath': str(matches[0].resolve())})
with LOCAL_SAMPLES_TSV.open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['sample-id', 'absolute-filepath'], delimiter='\t')
    writer.writeheader()
    writer.writerows(local_rows)
RUN_SAMPLES_TSV = LOCAL_SAMPLES_TSV
print(RUN_SAMPLES_TSV)

## Build and review the command

The options match the prior successful run recorded in `.nextflow/history`, with explicit current paths and output directory.

In [ ]:
RUN_SAMPLES_TSV = globals().get('RUN_SAMPLES_TSV', SAMPLES_TSV)
command = [
    'nextflow', 'run', 'main.nf',
    '--input', str(RUN_SAMPLES_TSV),
    '--metadata', str(METADATA_TSV),
    '--dada2_cpu', str(DADA2_CPUS),
    '--vsearch_cpu', str(VSEARCH_CPUS),
    '--outdir', str(OUTDIR),
    '-profile', PROFILE,
]
if RESUME:
    command.append('-resume')
print(shlex.join(command))

## Execute (explicit opt-in)

In [ ]:
if not RUN_PIPELINE:
    print('Dry run only. Set RUN_PIPELINE = True in the configuration cell, then rerun this cell.')
else:
    OUTDIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(command, cwd=PIPELINE_DIR, check=True)

## Inspect outputs

After completion, review `execution_report.html`, `execution_timeline.html`, `execution_trace.txt`, and the files under `results/`.

In [ ]:
if OUTDIR.exists():
    for path in sorted(OUTDIR.rglob('*')):
        if path.is_file():
            print(path.relative_to(OUTDIR))